# T0.7 Open API 응답 구조 확인 (api_probe)
법률 목록·본문, 시행령 본문, admrul/licbyl/lstrm/expc 목록을 1회씩 호출해 `data/raw/`에 저장하고 실제 키 구조를 확인한다.

- OC는 `common.config.LAW_OC`로만 주입. 저장 파일·manifest·출력에서는 OC 값을 `***`로 마스킹한다(목록 응답의 상세링크에 OC가 그대로 들어 있음).
- 정식 클라이언트는 T1.1의 `common/law_api.LawApiClient`. 이 노트북은 구조 확인용 일회성 프로브.

In [1]:
import hashlib
import json
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import httpx

from common.config import LAW_API_BASE, LAW_OC

assert LAW_OC, "LAW_OC가 .env에 없습니다"
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW = ROOT / "data" / "raw"
MANIFEST = RAW / "manifest.json"
_last_call = 0.0


def redact(text: str) -> str:
    """OC 값을 마스킹한다."""
    return text.replace(LAW_OC, "***")


def call(endpoint: str, oc: str | None = None, **params) -> httpx.Response:
    """DRF 엔드포인트 1회 호출 (호출 간 0.5s)."""
    global _last_call
    wait = 0.5 - (time.monotonic() - _last_call)
    if wait > 0:
        time.sleep(wait)
    r = httpx.get(f"{LAW_API_BASE}/{endpoint}", params={"OC": oc or LAW_OC, **params}, timeout=30, follow_redirects=True)
    _last_call = time.monotonic()
    return r


def save(r: httpx.Response, rel: str) -> dict:
    """OC를 마스킹한 원문을 data/raw/{rel}에 저장하고 manifest에 기록, JSON을 반환."""
    body = redact(r.text)
    path = RAW / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(body, encoding="utf-8")
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8")) if MANIFEST.exists() else {}
    manifest[rel] = {
        "url_without_oc": re.sub(r"OC=[^&]*&?", "", str(r.url)).rstrip("&?"),
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "bytes": len(body.encode("utf-8")),
        "sha256": hashlib.sha256(body.encode("utf-8")).hexdigest(),
        "content_type": r.headers.get("content-type"),
    }
    MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"saved {rel} ({manifest[rel]['bytes']:,} bytes, {r.headers.get('content-type')})")
    return json.loads(body)


def shape(x, depth=0, max_depth=3):
    """중첩 구조의 키·타입 요약."""
    pad = "  " * depth
    if isinstance(x, dict):
        for k, v in x.items():
            t = type(v).__name__ + (f"[{len(v)}]" if isinstance(v, (list, dict)) else "")
            preview = "" if isinstance(v, (list, dict)) else f" = {str(v)[:60]!r}"
            print(f"{pad}{k}: {t}{preview}")
            if depth < max_depth and isinstance(v, (dict, list)):
                shape(v, depth + 1, max_depth)
    elif isinstance(x, list) and x:
        print(f"{pad}[0] ->")
        shape(x[0], depth + 1, max_depth)


def as_list(x) -> list:
    return [] if x is None else x if isinstance(x, list) else [x]

## 1. 법률 목록 (target=law)

In [2]:
LAW_NAME = "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법"
r = call("lawSearch.do", target="law", type="JSON", query=LAW_NAME, display=100)
law_search = save(r, "law/_search_law.json")
print("root keys:", list(law_search), "| LawSearch keys:", list(law_search["LawSearch"]))
print("totalCnt:", law_search["LawSearch"]["totalCnt"], "| law type:", type(law_search["LawSearch"]["law"]).__name__)
for it in as_list(law_search["LawSearch"]["law"]):
    print(it["법령구분명"], "|", it["법령명한글"], "| MST", it["법령일련번호"], "| 법령ID", it["법령ID"], "| 시행", it["시행일자"], "| 공포", it["공포번호"], "|", it["현행연혁코드"])
print("\n목록 항목 키:", list(as_list(law_search["LawSearch"]["law"])[0]))

saved law/_search_law.json (1,973 bytes, application/json;charset=UTF-8)
root keys: ['LawSearch'] | LawSearch keys: ['law', 'resultMsg', '키워드', 'page', 'resultCode', 'target', 'totalCnt', 'section', 'numOfRows']
totalCnt: 2 | law type: list
법률 | 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 | MST 282791 | 법령ID 014820 | 시행 20260721 | 공포 21311 | 현행
대통령령 | 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 | MST 288781 | 법령ID 015032 | 시행 20260820 | 공포 36580 | 현행

목록 항목 키: ['현행연혁코드', '법령일련번호', '자법타법여부', '법령상세링크', '법령명한글', '법령구분명', '소관부처명', '공포번호', '제개정구분명', '소관부처코드', 'id', '법령ID', '공동부령정보', '시행일자', '공포일자', '법령약칭명']


## 2. 법률·시행령 본문 (lawService target=law MST=)

In [3]:
items = as_list(law_search["LawSearch"]["law"])
cur = [it for it in items if it["현행연혁코드"] == "현행"]
law_item = next(it for it in cur if it["법령구분명"] == "법률" and it["법령명한글"] == LAW_NAME)
decree_item = next(it for it in cur if it["법령구분명"] == "대통령령")
print("law MST", law_item["법령일련번호"], "| decree MST", decree_item["법령일련번호"], decree_item["법령명한글"])

law = save(call("lawService.do", target="law", type="JSON", MST=law_item["법령일련번호"]), f"law/{law_item['법령일련번호']}.json")
decree = save(call("lawService.do", target="law", type="JSON", MST=decree_item["법령일련번호"]), f"decree/{decree_item['법령일련번호']}.json")

law MST 282791 | decree MST 288781 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령


saved law/282791.json (156,580 bytes, application/json;charset=UTF-8)


saved decree/288781.json (180,836 bytes, application/json;charset=UTF-8)


In [4]:
print("법률 본문 root keys:", list(law))
shape(law, max_depth=1)

법률 본문 root keys: ['법령']
법령: dict[6]
  개정문: dict[1]
  법령키: str = '0148202026012021311'
  기본정보: dict[20]
  부칙: dict[1]
  조문: dict[1]
  제개정이유: dict[1]


In [5]:
def inspect_body(doc: dict, label: str) -> None:
    """조문단위·항·호·목의 타입 분포와 특이 케이스를 출력."""
    root = doc["법령"]
    print(f"===== {label}: {root['기본정보'].get('법령명_한글')} =====")
    print("기본정보 keys:", list(root["기본정보"]))
    units = as_list(root["조문"]["조문단위"])
    print("조문 keys:", list(root["조문"]), "| 조문단위 type:", type(root["조문"]["조문단위"]).__name__, len(units))
    print("조문여부 분포:", Counter(u.get("조문여부") for u in units))
    print("조문단위 key 합집합:", sorted({k for u in units for k in u}))
    hang_t, ho_t, mok_t, content_t = Counter(), Counter(), Counter(), Counter()
    no_hang, branch = [], []
    for u in units:
        if u.get("조문여부") != "조문":
            continue
        content_t[type(u.get("조문내용")).__name__] += 1
        if u.get("조문가지번호"):
            branch.append(f"제{u['조문번호']}조의{u['조문가지번호']}")
        if "항" not in u:
            no_hang.append(u["조문번호"])
            continue
        hang_t[type(u["항"]).__name__] += 1
        for h in as_list(u["항"]):
            if "호" in h:
                ho_t[type(h["호"]).__name__] += 1
                for ho in as_list(h["호"]):
                    if "목" in ho:
                        mok_t[type(ho["목"]).__name__] += 1
                        for mk in as_list(ho["목"]):
                            content_t["목내용:" + type(mk.get("목내용")).__name__] += 1
    print("항 type:", dict(hang_t), "| 호 type:", dict(ho_t), "| 목 type:", dict(mok_t))
    print("내용 필드 type:", dict(content_t))
    print("항 없는 조:", no_hang)
    print("가지조문:", branch)
    print("부칙 keys:", list(root.get("부칙", {})), "| 별표 keys:", list(root.get("별표", {})))


inspect_body(law, "법률")
print()
inspect_body(decree, "시행령")

===== 법률: 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 =====
기본정보 keys: ['법령명_한글', '공포번호', '전화번호', '언어', '제개정구분', '법령ID', '공동부령정보', '소관부처', '공포법령여부', '법종구분', '제명변경여부', '시행일자', '별표편집여부', '연락부서', '조문시행일자문자열', '법령명_한자', '법령명약칭', '공포일자', '한글법령여부', '편장절관']
조문 keys: ['조문단위'] | 조문단위 type: list 54
조문여부 분포: Counter({'조문': 46, '전문': 8})
조문단위 key 합집합: ['조문가지번호', '조문내용', '조문번호', '조문변경여부', '조문시행일자', '조문여부', '조문이동이전', '조문이동이후', '조문제개정유형', '조문제개정일자문자열', '조문제목', '조문참고자료', '조문키', '항']
항 type: {'dict': 2, 'list': 41} | 호 type: {'list': 34} | 목 type: {'list': 2}
내용 필드 type: {'str': 46, '목내용:str': 13}
항 없는 조: ['1', '22', '42']
가지조문: ['제17조의2', '제22조의2', '제22조의3']
부칙 keys: ['부칙단위'] | 별표 keys: []

===== 시행령: 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 =====
기본정보 keys: ['법령명_한글', '공포번호', '전화번호', '언어', '제개정구분', '법령ID', '공동부령정보', '소관부처', '공포법령여부', '법종구분', '제명변경여부', '시행일자', '별표편집여부', '연락부서', '법령명_한자', '법령명약칭', '공포일자', '한글법령여부', '편장절관']
조문 keys: ['조문단위'] | 조문단위 type: list 46
조문여부 분포: Counter({'조문': 39, '전문': 7})
조문단위 key 합집합: ['조문가지번호', '조문

In [6]:
# 대표 조문 샘플: 전문 행, 항이 dict(단건)인 조, 항이 list인 조, 호·목이 있는 조
units = as_list(law["법령"]["조문"]["조문단위"])
jeonmun = next(u for u in units if u["조문여부"] == "전문")
print("[전문 행]", json.dumps(jeonmun, ensure_ascii=False)[:400])
single = next((u for u in units if u["조문여부"] == "조문" and isinstance(u.get("항"), dict)), None)
print("\n[항 dict 단건]", json.dumps(single, ensure_ascii=False)[:600] if single else "없음")
nohang = next((u for u in units if u["조문여부"] == "조문" and "항" not in u), None)
print("\n[항 없음]", json.dumps(nohang, ensure_ascii=False)[:400] if nohang else "없음")
art2 = next(u for u in units if u["조문여부"] == "조문" and u["조문번호"] == "2")
print("\n[제2조 (호·목)]", json.dumps(art2, ensure_ascii=False)[:1200])

[전문 행] {"조문번호": "1", "조문시행일자": "20260122", "조문변경여부": "N", "조문이동이전": "", "조문키": "0001000", "조문내용": "            제1장 총칙", "조문이동이후": "", "조문여부": "전문"}

[항 dict 단건] {"조문번호": "2", "조문제개정유형": "개정", "조문시행일자": "20260122", "조문변경여부": "Y", "조문이동이전": "000000", "조문키": "0002001", "조문제개정일자문자열": "2026.1.20", "항": {"호": [{"호번호": "1.", "호내용": "1. \"인공지능\"이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다."}, {"호번호": "2.", "호내용": "2. \"인공지능시스템\"이란 다양한 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향을 미치는 예측, 추천, 결정 등의 결과물을 추론하는 인공지능 기반 시스템을 말한다."}, {"호번호": "3.", "호내용": "3. \"인공지능기술\"이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다."}, {"호번호": "4.", "호내용": "4. \"고영향 인공지능\"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을

[항 없음] {"조문번호": "1", "조문시행일자": "20260122", "조문변경여부": "N", "조문이동이전": "", "조문키": "0001001", "조문내용": "제1조(목적) 이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다.", "조문제목": "목적", "조문이동

In [7]:
art31 = next(u for u in units if u["조문여부"] == "조문" and u["조문번호"] == "31")
print("[제31조]", json.dumps(art31, ensure_ascii=False, indent=1)[:1500])

[제31조] {
 "조문번호": "31",
 "조문시행일자": "20260122",
 "조문변경여부": "N",
 "조문이동이전": "",
 "조문키": "0031001",
 "항": [
  {
   "항번호": "①",
   "항내용": "① 인공지능사업자는 고영향 인공지능이나 생성형 인공지능을 이용한 제품 또는 서비스를 제공하려는 경우 제품 또는 서비스가 해당 인공지능에 기반하여 운용된다는 사실을 이용자에게 사전에 고지하여야 한다."
  },
  {
   "항번호": "②",
   "항내용": "② 인공지능사업자는 생성형 인공지능 또는 이를 이용한 제품 또는 서비스를 제공하는 경우 그 결과물이 생성형 인공지능에 의하여 생성되었다는 사실을 표시하여야 한다."
  },
  {
   "항번호": "③",
   "항내용": "③ 인공지능사업자는 인공지능시스템을 이용하여 실제와 구분하기 어려운 가상의 음향, 이미지 또는 영상 등의 결과물을 제공하는 경우 해당 결과물이 인공지능시스템에 의하여 생성되었다는 사실을 이용자가 명확하게 인식할 수 있는 방식으로 고지 또는 표시하여야 한다. 이 경우 해당 결과물이 예술적ㆍ창의적 표현물에 해당하거나 그 일부를 구성하는 경우에는 전시 또는 향유 등을 저해하지 아니하는 방식으로 고지 또는 표시할 수 있다."
  },
  {
   "항번호": "④",
   "항내용": "④ 그 밖에 제1항에 따른 사전고지, 제2항에 따른 표시, 제3항에 따른 고지 또는 표시의 방법 및 그 예외 등에 관하여 필요한 사항은 대통령령으로 정한다."
  }
 ],
 "조문내용": "제31조(인공지능 투명성 확보 의무)",
 "조문제목": "인공지능 투명성 확보 의무",
 "조문이동이후": "",
 "조문여부": "조문"
}


In [8]:
for label, doc in [("법률", law), ("시행령", decree)]:
    root = doc["법령"]
    print(f"== {label} 부칙 ==")
    for b in as_list(root.get("부칙", {}).get("부칙단위")):
        print({k: (type(v).__name__ if isinstance(v, (list, dict)) else str(v)[:40]) for k, v in b.items()})
    print(f"== {label} 별표 ==")
    for b in as_list(root.get("별표", {}).get("별표단위")):
        print({k: (type(v).__name__ if isinstance(v, (list, dict)) else str(v)[:70]) for k, v in b.items()})

== 법률 부칙 ==
{'부칙키': '2025012120676', '부칙공포일자': '20250121', '부칙내용': 'list', '부칙공포번호': '20676'}
{'부칙키': '2025100121065', '부칙공포일자': '20251001', '부칙내용': 'list', '부칙공포번호': '21065'}
{'부칙키': '2026012021311', '부칙공포일자': '20260120', '부칙내용': 'list', '부칙공포번호': '21311'}
== 법률 별표 ==
== 시행령 부칙 ==
{'부칙키': '2026012136053', '부칙공포일자': '20260121', '부칙내용': 'list', '부칙공포번호': '36053'}
{'부칙키': '2026072036506', '부칙공포일자': '20260720', '부칙내용': 'list', '부칙공포번호': '36506'}
{'부칙키': '2026081836580', '부칙공포일자': '20260818', '부칙내용': 'list', '부칙공포번호': '36580'}
== 시행령 별표 ==
{'별표제목': '이행조치 인정 기준 및 절차(제27조제5항 관련)', '별표서식이미지파일링크': 'list', '별표PDF파일명': 'law0150322026081836580KC_000100E.pdf', '별표HWP파일명': 'law0150322026081836580KC_000100E.hwp', '별표번호': '0001', '별표서식PDF파일링크': '/LSW/flDownload.do?flSeq=168236245', '별표키': '000100E', '별표내용': 'list', '별표이미지파일명': 'list', '별표구분': '별표', '별표서식파일링크': '/LSW/flDownload.do?flSeq=168236243', '별표가지번호': '00'}
{'별표제목': '과태료의 부과기준(제32조 관련)', '별표서식이미지파일링크': 'list', '별표PDF파일명': 'law015032202608183658

## 3. 행정규칙 목록 (target=admrul)

In [9]:
r = call("lawSearch.do", target="admrul", type="JSON", query="인공지능", display=100)
print(r.status_code, r.headers.get("content-type"))
admrul = save(r, "admrul/_search_인공지능.json")
print("root:", list(admrul))
s = next(iter(admrul.values()))
print("keys:", list(s), "| totalCnt:", s.get("totalCnt"))
rows = as_list(s.get("admrul"))
if rows:
    print("항목 키:", list(rows[0]))
    for it in rows:
        print(it.get("소관부처명"), "|", it.get("행정규칙종류"), "|", it.get("행정규칙명"), "| ID", it.get("행정규칙일련번호"), "| 발령", it.get("발령일자"), "|", it.get("현행연혁구분"))

200 application/json;charset=UTF-8
saved admrul/_search_인공지능.json (20,846 bytes, application/json;charset=UTF-8)
root: ['AdmRulSearch']
keys: ['resultMsg', '키워드', 'page', 'resultCode', 'target', 'admrul', 'totalCnt', 'section', 'numOfRows'] | totalCnt: 27
항목 키: ['현행연혁구분', '행정규칙명', '발령일자', '행정규칙종류', '소관부처명', '제개정구분코드', '행정규칙상세링크', '제개정구분명', 'id', '생성일자', '행정규칙ID', '시행일자', '발령번호', '행정규칙일련번호']
경찰청 | 훈령 | 경찰청 인공지능의 개발 및 활용에 관한 규칙 | ID 2100000281962 | 발령 20260630 | 현행
과학기술정보통신부 | 국무총리훈령 | 국민인공지능서비스혁신추진단의 설치 및 운영에 관한 규정 | ID 2200000108879 | 발령 20260730 | 현행
국방부 | 훈령 | 국방데이터·인공지능업무 훈령 | ID 2100000276642 | 발령 20260330 | 현행
국무조정실 | 국무총리훈령 | 글로벌 인공지능 허브 추진위원회의 설치 및 운영에 관한 규정 | ID 2200000108951 | 발령 20260909 | 현행
소방청 | 훈령 | 소방 인공지능·로봇 기술위원회 운영 규정 | ID 2100000281220 | 발령 20260630 | 현행
식품의약품안전처 | 훈령 | 식·의약 인공지능 전환 추진단 운영 규정 | ID 2100000274448 | 발령 20260204 | 현행
과학기술정보통신부 | 고시 | 인공지능제품ㆍ서비스 확인 절차 운영에 관한 고시 | ID 2100000283290 | 발령 20260721 | 현행
과학기술정보통신부 | 훈령 | 자율기구 과학기술·인공지능미래전략담당관 설치 및 운영에 관한 규정 | I

## 4. 별표·서식 목록 (target=licbyl)

In [10]:
r = call("lawSearch.do", target="licbyl", type="JSON", query=LAW_NAME, display=100)
print(r.status_code, r.headers.get("content-type"))
licbyl = save(r, "annex/_search_licbyl.json")
print("root:", list(licbyl))
s = next(iter(licbyl.values()))
print("keys:", list(s), "| totalCnt:", s.get("totalCnt"))
rows = as_list(s.get("licbyl"))
if rows:
    print("항목 키:", list(rows[0]))
    for it in rows:
        print(it.get("관련법령명"), "|", it.get("별표종류"), it.get("별표번호"), "|", it.get("별표명"))
    print("\n첫 항목:", json.dumps(rows[0], ensure_ascii=False, indent=1))

200 application/json;charset=UTF-8
saved annex/_search_licbyl.json (268 bytes, application/json;charset=UTF-8)
root: ['licBylSearch']
keys: ['resultMsg', '키워드', 'page', 'resultCode', 'target', 'totalCnt', 'section', 'numOfRows'] | totalCnt: 0


## 5. 법령용어 목록 (target=lstrm)

In [11]:
r = call("lawSearch.do", target="lstrm", type="JSON", query="인공지능", display=100)
print(r.status_code, r.headers.get("content-type"))
lstrm = save(r, "term/_search_인공지능.json")
print("root:", list(lstrm))
s = next(iter(lstrm.values()))
print("keys:", list(s), "| totalCnt:", s.get("totalCnt"))
rows = as_list(s.get("lstrm"))
if rows:
    print("항목 키:", list(rows[0]))
    for it in rows[:20]:
        print(json.dumps(it, ensure_ascii=False)[:200])

200 application/json;charset=UTF-8
saved term/_search_인공지능.json (10,239 bytes, application/json;charset=UTF-8)
root: ['LsTrmSearch']
keys: ['resultMsg', '키워드', 'page', 'resultCode', 'target', 'lstrm', 'totalCnt', 'section', 'numOfRows'] | totalCnt: 21
항목 키: ['id', '법령용어상세링크', '법령종류코드', '법령용어ID', '법령용어명', '법령용어상세검색', '사전구분코드']
{"id": "1", "법령용어상세링크": "/DRF/lawService.do?OC=***&target=lstrm&trmSeqs=5513400,5512250&mobile=&type=XML", "법령종류코드": "010102,010101", "법령용어ID": "5513400,5512250", "법령용어명": "고영향 인공지능", "법령용어상세검색": "/LS
{"id": "2", "법령용어상세링크": "/DRF/lawService.do?OC=***&target=lstrm&trmSeqs=5542198&mobile=&type=XML", "법령종류코드": "010101", "법령용어ID": "5542198", "법령용어명": "산업 디지털 전환 및 인공지능 활용", "법령용어상세검색": "/LSW/lsTrmInfoR
{"id": "3", "법령용어상세링크": "/DRF/lawService.do?OC=***&target=lstrm&trmSeqs=5542195&mobile=&type=XML", "법령종류코드": "010101", "법령용어ID": "5542195", "법령용어명": "산업인공지능", "법령용어상세검색": "/LSW/lsTrmInfoR.do?trmSeqs=5
{"id": "4", "법령용어상세링크": "/DRF/lawService.do?OC=***&target=lstrm&trmSe

## 6. 법령해석례 목록 (target=expc)

In [12]:
r = call("lawSearch.do", target="expc", type="JSON", query="인공지능", display=100)
print(r.status_code, r.headers.get("content-type"))
expc = save(r, "expc/_search_인공지능.json")
print("root:", list(expc))
s = next(iter(expc.values()))
print("keys:", list(s), "| totalCnt:", s.get("totalCnt"))
rows = as_list(s.get("expc"))
if rows:
    print("항목 키:", list(rows[0]))
    for it in rows[:20]:
        print(it.get("안건명"), "| ID", it.get("법령해석례일련번호"), "|", it.get("회신기관명") or it.get("해석기관명"), it.get("회신일자"))

200 application/json;charset=UTF-8
saved expc/_search_인공지능.json (203 bytes, application/json;charset=UTF-8)
root: ['Expc']
keys: ['resultMsg', '키워드', 'page', 'resultCode', 'target', 'totalCnt', 'section', 'numOfRows'] | totalCnt: 0


## 7. 인증 실패 시 응답 (잘못된 OC) — 저장하지 않음

In [13]:
bad = call("lawSearch.do", oc="invalid-oc-probe", target="law", type="JSON", query="인공지능")
print("status:", bad.status_code, "| content-type:", bad.headers.get("content-type"))
print(redact(bad.text)[:500])

status: 200 | content-type: application/json;charset=UTF-8
{
    "result" : "사용자 정보 검증에 실패하였습니다.",
    "msg" : "OPEN API 호출 시 사용자 검증을 위하여 정확한 서버장비의 IP주소 및 도메인주소를 등록해 주세요."
}



## 8. 추가 확인
### 8-1. licbyl: 법령명으로 검색하면 0건 → `search=2`(해당법령 검색)로 재시도

In [14]:
DECREE_NAME = LAW_NAME + " 시행령"
for q in [LAW_NAME, DECREE_NAME]:
    r = call("lawSearch.do", target="licbyl", type="JSON", query=q, search=2, display=100)
    s = next(iter(r.json().values()))
    print(q, "-> totalCnt", s.get("totalCnt"))
    if int(s.get("totalCnt") or 0):
        licbyl = save(r, "annex/_search_licbyl_search2_" + ("decree" if q.endswith("시행령") else "law") + ".json")
        rows = as_list(next(iter(licbyl.values())).get("licbyl"))
        print("항목 키:", list(rows[0]))
        for it in rows:
            print(" ", it.get("관련법령명"), "|", it.get("별표종류"), it.get("별표번호"), "|", it.get("별표명"))
        print("첫 항목:", json.dumps(rows[0], ensure_ascii=False, indent=1))

인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 -> totalCnt 3
saved annex/_search_licbyl_search2_law.json (3,153 bytes, application/json;charset=UTF-8)
항목 키: ['관련법령명', '관련법령ID', '소관부처명', '공포번호', '별표서식파일링크', '법령종류', '제개정구분명', 'id', '별표일련번호', '별표법령상세링크', '별표종류', '별표번호', '별표서식PDF파일링크', '별표명', '관련법령일련번호', '공포일자']
  인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 | 서식 000000 | 고영향 인공지능 (확인, 재확인)요청서
  인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 | 별표 000200 | 과태료의 부과기준(제32조 관련)
  인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 | 별표 000100 | 이행조치 인정 기준 및 절차(제27조제5항 관련)
첫 항목: {
 "관련법령명": "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령",
 "관련법령ID": "015032",
 "소관부처명": "과학기술정보통신부",
 "공포번호": "36580",
 "별표서식파일링크": "/LSW/flDownload.do?flSeq=168105407",
 "법령종류": "대통령령",
 "제개정구분명": "타법개정",
 "id": "1",
 "별표일련번호": "18377895",
 "별표법령상세링크": "/DRF/lawService.do?OC=***&target=licbyl&ID=18377895&type=HTML&mobileYn=",
 "별표종류": "서식",
 "별표번호": "000000",
 "별표서식PDF파일링크": "/LSW/flDownload.do?flSeq=168105409",
 "별표명": "고영향 인공지능 (확인, 재확인)요청서",
 "관련법령일련번호": "288781",
 "공포일자": "20260818"
}


인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 -> totalCnt 3
saved annex/_search_licbyl_search2_decree.json (3,163 bytes, application/json;charset=UTF-8)
항목 키: ['관련법령명', '관련법령ID', '소관부처명', '공포번호', '별표서식파일링크', '법령종류', '제개정구분명', 'id', '별표일련번호', '별표법령상세링크', '별표종류', '별표번호', '별표서식PDF파일링크', '별표명', '관련법령일련번호', '공포일자']
  인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 | 서식 000000 | 고영향 인공지능 (확인, 재확인)요청서
  인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 | 별표 000200 | 과태료의 부과기준(제32조 관련)
  인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 | 별표 000100 | 이행조치 인정 기준 및 절차(제27조제5항 관련)
첫 항목: {
 "관련법령명": "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령",
 "관련법령ID": "015032",
 "소관부처명": "과학기술정보통신부",
 "공포번호": "36580",
 "별표서식파일링크": "/LSW/flDownload.do?flSeq=168105407",
 "법령종류": "대통령령",
 "제개정구분명": "타법개정",
 "id": "1",
 "별표일련번호": "18377895",
 "별표법령상세링크": "/DRF/lawService.do?OC=***&target=licbyl&ID=18377895&type=HTML&mobileYn=",
 "별표종류": "서식",
 "별표번호": "000000",
 "별표서식PDF파일링크": "/LSW/flDownload.do?flSeq=168105409",
 "별표명": "고영향 인공지능 (확인, 재확인)요청서",
 "관련법령일련번호": "288781",
 "공포일자": "20260818"


### 8-2. 시행령 본문 `별표단위`의 `별표내용` 형태 (본문 JSON에 별표 텍스트가 들어 있는가)

In [15]:
for b in as_list(decree["법령"]["별표"]["별표단위"]):
    c = b["별표내용"]
    print(f"== 별표 {b['별표번호']} ({b['별표구분']}) {b['별표제목']} | 별표내용 type={type(c).__name__}, len={len(c)}, [0] type={type(c[0]).__name__}")
    flat = c
    while flat and isinstance(flat[0], list):
        flat = [y for x in flat for y in x]
    text = "\n".join(map(str, flat))
    print(text[:700])
    print("... 총", len(text), "자\n")

== 별표 0001 (별표) 이행조치 인정 기준 및 절차(제27조제5항 관련) | 별표내용 type=list, len=1, [0] type=list
■ 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 [별표 1]            
  [시행일: 2026. 1. 24.] 디지털의료기기에 관한 부분                             
                                                                                
  이행조치 인정 기준 및 절차(제27조제5항 관련)                                  
                                                                                
                                                                                
  1. 인공지능사업자가 개발하거나 제공하는 인공지능제품 또는 인공지능서비스      
  가 다음 각 목의 어느 하나에 해당하는 경우에는 법 제34조제1항제1호에 따        
  른 조치를 이행한 것으로 본다.                                                 
    가. 법 제2조제4호라목의 「디지털의료제품법」 제2조제2호에 따른 디지털의     
  료기기(이하 "디지털의료기기"라 한다): 같은 법 제8조제4항 또는 같은 법        
... 총 5884 자

== 별표 0002 (별표) 과태료의 부과기준(제32조 관련) | 별표내용 type=list, len=1, [0] type=list
■ 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령 [별표 2]              
                                                                         

### 8-3. 항내용·호내용·부칙내용 타입

In [16]:
def types_of(doc):
    t = Counter()
    for u in as_list(doc["법령"]["조문"]["조문단위"]):
        for h in as_list(u.get("항")):
            if "항내용" in h:
                t["항내용:" + type(h["항내용"]).__name__] += 1
            else:
                t["항내용 없음(호만)"] += 1
            for ho in as_list(h.get("호")):
                t["호내용:" + type(ho.get("호내용")).__name__] += 1
    for b in as_list(doc["법령"]["부칙"]["부칙단위"]):
        c = b["부칙내용"]
        t[f"부칙내용:{type(c).__name__}[{type(c[0]).__name__ if c else '-'}]"] += 1
    return dict(t)


print("법률:", types_of(law))
print("시행령:", types_of(decree))
nested = [h for u in as_list(decree["법령"]["조문"]["조문단위"]) for h in as_list(u.get("항")) if isinstance(h.get("항내용"), list)]
if nested:
    print("중첩 리스트 항내용 예:", json.dumps(nested[0]["항내용"], ensure_ascii=False)[:300])
print("부칙내용 예:", json.dumps(law["법령"]["부칙"]["부칙단위"][0]["부칙내용"], ensure_ascii=False)[:300])

법률: {'항내용 없음(호만)': 2, '호내용:str': 191, '항내용:str': 164, '부칙내용:list[list]': 3}
시행령: {'항내용 없음(호만)': 4, '호내용:str': 221, '항내용:str': 116, '부칙내용:list[list]': 3}
부칙내용 예: [["부칙 <제20676호,2025.1.21>", "제1조(시행일) 이 법은 공포 후 1년이 경과한 날부터 시행한다. 다만, 제2조제4호라목 중 디지털의료기기에 관한 부분은 2026년 1월 24일부터 시행한다.", "제2조(이 법 시행을 위한 준비행위) 이 법을 시행하기 위하여 필요한 위원회 위원의 위촉, 분과위원회, 특별위원회, 자문단 및 지원단의 구성 등은 이 법 시행 전에 할 수 있다.", "제3조(전담기관에 관한 특례) 이 법 시행 당시 제23조제1항에 따른 집적화를 지역에 효과적으로 정착시키기 위하여 정부로부터 관련 예산을


### 8-4. 별표 HTML 상세 (`별표법령상세링크`, type=HTML) — 별표 2 과태료 부과기준 1건

In [17]:
rows = as_list(next(iter(licbyl.values())).get("licbyl"))
target = next(it for it in rows if it["별표번호"] == "000200")
print("링크(마스킹됨):", target["별표법령상세링크"])
# 저장된 링크는 마스킹돼 있으므로 ID로 다시 조립해 OC를 주입한다
r = call("lawService.do", target="licbyl", ID=target["별표일련번호"], type="HTML")
print(r.status_code, r.headers.get("content-type"), len(r.text), "chars")
path = RAW / "annex" / f"{target['별표일련번호']}.html"
path.write_text(redact(r.text), encoding="utf-8")
print("saved", path.relative_to(ROOT))
print(redact(r.text)[:600])

링크(마스킹됨): /DRF/lawService.do?OC=***&target=licbyl&ID=18377893&type=HTML&mobileYn=


200 text/html;charset=UTF-8 3152 chars
saved data/raw/annex/18377893.html






















<html>
<head>
<style type="text/css">
	body {
	    overflow: hidden; 
	    width: 100%;
		height : 100%;
	}
</style>
<script type="text/javascript">
	function onLoad(){
		getNwRvsList();
	}
</script>
</head>
<body topmargin="0" leftmargin="0" rightmargin="0" bottommargin="0" marginheight="0" marginwidth="0" onLoad="onLoad();">
<table cellspacing="1" cellpadding="0" border="0" height="98%" width="100%" style="border:1px solid #cacaca;">
<tr>
<td>

<iframe	id="bylService"
			name="bylService"
	      	scrolling="auto" frameborder="
